In [1]:
import os, random, torch, numpy as np


In [2]:
from PIL import Image
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

DATASET_PATH = r"G:\Ragib\Prescription\Full_Dataset_Split"
BATCH_SIZE = 16

train_tf = transforms.Compose([
    transforms.RandomRotation(10),
    transforms.RandomAffine(0, translate=(0.05,0.05)),
    transforms.ColorJitter(contrast=0.2),
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])

val_tf = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
])


def is_valid_image(path):
    try:
        if os.path.getsize(path) == 0:
            return False
        with Image.open(path) as img:
            img.verify()
        return True
    except Exception:
        return False


class SafeImageFolder(datasets.ImageFolder):
    """ImageFolder that skips unreadable or zero-byte image files."""

    def __init__(self, root, transform=None, target_transform=None):
        super().__init__(root, transform=transform, target_transform=target_transform)
        valid_samples = []
        skipped = []
        for path, target in self.samples:
            if is_valid_image(path):
                valid_samples.append((path, target))
            else:
                skipped.append(path)
        if skipped:
            print(f"Skipping {len(skipped)} invalid image(s) in {root}:")
            for path in skipped:
                print(f"  - {path}")
        self.samples = valid_samples
        self.imgs = self.samples
        self.targets = [target for _, target in valid_samples]


train_ds = SafeImageFolder(os.path.join(DATASET_PATH, "Train"), train_tf)
val_ds   = SafeImageFolder(os.path.join(DATASET_PATH, "Val"), val_tf)
test_ds  = SafeImageFolder(os.path.join(DATASET_PATH, "Test"), val_tf)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

NUM_CLASSES = len(train_ds.classes)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


Skipping 1 invalid image(s) in G:\Ragib\Prescription\Full_Dataset_Split\Test:
  - G:\Ragib\Prescription\Full_Dataset_Split\Test\Tufnil\38.png


In [3]:
import torch
import torch.nn as nn
from tqdm import tqdm

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

def train_epoch(model, optimizer, train_loader, device):
    model.train()
    loss_sum, correct = 0, 0

    for x, y in tqdm(train_loader, leave=False):
        x, y = x.to(device), y.to(device)
        optimizer.zero_grad()

        out = model(x)
        loss = criterion(out, y)
        loss.backward()
        optimizer.step()

        loss_sum += loss.item()
        correct += (out.argmax(1) == y).sum().item()

    return loss_sum / len(train_loader), correct / len(train_loader.dataset)


def eval_epoch(model, val_loader, device):
    model.eval()
    loss_sum, correct = 0, 0

    with torch.no_grad():
        for x, y in val_loader:
            x, y = x.to(device), y.to(device)
            out = model(x)
            loss = criterion(out, y)

            loss_sum += loss.item()
            correct += (out.argmax(1) == y).sum().item()

    return loss_sum / len(val_loader), correct / len(val_loader.dataset)


In [4]:
import json
import time
from torchvision import models
import torch.nn as nn

# =========================
# MULTI-RUN CONFIGURATION
# =========================
NUM_RUNS = 3
RANDOM_SEEDS = [42, 123, 456]
OUTPUT_DIR = "multirun_baseline"
EPOCHS = 25

BASELINE_MODELS = [
    ("vgg16", lambda: models.vgg16(pretrained=True)),
    ("vgg19", lambda: models.vgg19(pretrained=True)),
    ("densenet121", lambda: models.densenet121(pretrained=True)),
    ("mobilenetv2", lambda: models.mobilenet_v2(pretrained=True)),
    ("efficientnetb0", lambda: models.efficientnet_b0(pretrained=True)),
]

def configure_baseline_model(name, model):
    if name in ("vgg16", "vgg19"):
        model.classifier[6] = nn.Linear(4096, NUM_CLASSES)
    elif name == "densenet121":
        model.classifier = nn.Linear(1024, NUM_CLASSES)
    elif name == "mobilenetv2":
        model.classifier[1] = nn.Linear(1280, NUM_CLASSES)
    elif name == "efficientnetb0":
        model.classifier[1] = nn.Linear(1280, NUM_CLASSES)
    return model


def train_eval_baseline_model(MODEL_NAME, model_builder, run_dir, device):
    import torch.optim as optim
    import matplotlib.pyplot as plt
    import seaborn as sns
    from sklearn.metrics import (
        accuracy_score,
        precision_recall_fscore_support,
        confusion_matrix,
    )

    model = model_builder().to(device)
    optimizer = optim.AdamW(model.parameters(), lr=1e-4)

    CHECKPOINT = os.path.join(run_dir, f"checkpoint_{MODEL_NAME}.pth")
    BEST_MODEL = os.path.join(run_dir, f"best_{MODEL_NAME}.pth")

    start_epoch = 0
    best_val_acc = 0.0
    best_epoch = 0
    train_losses, val_losses = [], []
    train_accs, val_accs = [], []

    if os.path.exists(CHECKPOINT):
        try:
            ckpt = torch.load(CHECKPOINT, map_location=device)
            model.load_state_dict(ckpt["model"])
            optimizer.load_state_dict(ckpt["optimizer"])
            start_epoch = ckpt["epoch"] + 1
            best_val_acc = ckpt["best_val_acc"]
            best_epoch = ckpt.get("best_epoch", start_epoch)
            train_losses = ckpt["train_losses"]
            val_losses   = ckpt["val_losses"]
            train_accs   = ckpt["train_accs"]
            val_accs     = ckpt["val_accs"]
            print("✅ Resumed from checkpoint")
        except Exception as e:
            print("⚠️ Checkpoint corrupted. Starting fresh.")
            print(e)

    train_start = time.time()

    for epoch in range(start_epoch, EPOCHS):
        tr_l, tr_a = train_epoch(model, optimizer, train_loader, device)
        vl_l, vl_a = eval_epoch(model, val_loader, device)

        train_losses.append(tr_l)
        val_losses.append(vl_l)
        train_accs.append(tr_a)
        val_accs.append(vl_a)

        if vl_a > best_val_acc:
            best_val_acc = vl_a
            best_epoch = epoch + 1
            torch.save(model.state_dict(), BEST_MODEL)

        torch.save({
            "epoch": epoch,
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "best_val_acc": best_val_acc,
            "best_epoch": best_epoch,
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs
        }, CHECKPOINT)

        print(
            f"[{MODEL_NAME}] Epoch {epoch+1}/{EPOCHS} | "
            f"Train Loss: {tr_l:.4f} | Train Acc: {tr_a:.4f} | "
            f"Val Loss: {vl_l:.4f} | Val Acc: {vl_a:.4f}"
        )

    training_time = time.time() - train_start

    plt.figure()
    plt.plot(train_accs, label="Train Acc")
    plt.plot(val_accs, label="Val Acc")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend()
    plt.grid()
    plt.title(f"{MODEL_NAME} Accuracy Curve")
    plt.savefig(os.path.join(run_dir, f"{MODEL_NAME}_acc.png"), dpi=300)
    plt.close()

    plt.figure()
    plt.plot(train_losses, label="Train Loss")
    plt.plot(val_losses, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.grid()
    plt.title(f"{MODEL_NAME} Loss Curve")
    plt.savefig(os.path.join(run_dir, f"{MODEL_NAME}_loss.png"), dpi=300)
    plt.close()

    model.load_state_dict(torch.load(BEST_MODEL, map_location=device))
    model.eval()

    y_true, y_pred, y_pred_top5 = [], [], []

    with torch.no_grad():
        for x, y in test_loader:
            x = x.to(device)
            probs = torch.softmax(model(x), dim=1)

            y_true.extend(y.numpy())
            y_pred.extend(probs.argmax(1).cpu().numpy())
            y_pred_top5.extend(torch.topk(probs, 5, dim=1).indices.cpu().numpy())

    acc = accuracy_score(y_true, y_pred)
    top5 = sum(
        y_true[i] in y_pred_top5[i]
        for i in range(len(y_true))
    ) / len(y_true)

    p_macro, r_macro, f1_macro, _ = precision_recall_fscore_support(
        y_true, y_pred, average="macro", zero_division=0
    )
    p_weighted, r_weighted, f1_weighted, _ = precision_recall_fscore_support(
        y_true, y_pred, average="weighted", zero_division=0
    )

    print(f"\n📊 {MODEL_NAME.upper()} TEST RESULTS")
    print(f"Accuracy      : {acc:.4f}")
    print(f"Top-5 Acc     : {top5:.4f}")
    print(f"Macro F1      : {f1_macro:.4f}")
    print(f"Weighted F1   : {f1_weighted:.4f}")

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(8,6))
    sns.heatmap(cm, cmap="Blues", xticklabels=False, yticklabels=False)
    plt.title(f"{MODEL_NAME} Confusion Matrix")
    plt.savefig(os.path.join(run_dir, f"{MODEL_NAME}_confusion.png"), dpi=300)
    plt.close()

    with open(os.path.join(run_dir, f"{MODEL_NAME}_training_history.json"), "w", encoding="utf-8") as f:
        json.dump({
            "train_losses": train_losses,
            "val_losses": val_losses,
            "train_accs": train_accs,
            "val_accs": val_accs,
        }, f, indent=2)

    del model
    torch.cuda.empty_cache()

    return {
        "Model": MODEL_NAME,
        "Top-1 Accuracy": acc,
        "Top-5 Accuracy": top5,
        "Precision": p_macro,
        "Recall": r_macro,
        "Macro F1": f1_macro,
        "Weighted F1": f1_weighted,
        "Training Time": training_time,
        "Best Epoch": best_epoch,
    }


os.makedirs(OUTPUT_DIR, exist_ok=True)
all_run_results = []

for run_idx in range(NUM_RUNS):
    run_num = run_idx + 1
    seed = RANDOM_SEEDS[run_idx]
    run_dir = os.path.join(OUTPUT_DIR, f"run_{run_num}")
    os.makedirs(run_dir, exist_ok=True)

    complete_flag = os.path.join(run_dir, "run_complete.json")
    if os.path.exists(complete_flag):
        print(f"\n✅ Run {run_num} already complete. Loading saved results.")
        with open(complete_flag, "r", encoding="utf-8") as f:
            saved = json.load(f)
            all_run_results.extend(saved)
        continue

    print(f"\n{'='*60}")
    print(f"Starting Run {run_num}/{NUM_RUNS} | Seed: {seed}")
    print(f"{'='*60}")

    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    os.environ["PYTHONHASHSEED"] = str(seed)

    run_results = []

    for model_name, model_fn in BASELINE_MODELS:
        print(f"\n--- Training {model_name} ---")

        def build_model(model_fn=model_fn, model_name=model_name):
            return configure_baseline_model(model_name, model_fn())

        result = train_eval_baseline_model(model_name, build_model, run_dir, DEVICE)
        result["Run Number"] = run_num
        result["Random Seed"] = seed
        run_results.append(result)

    with open(complete_flag, "w", encoding="utf-8") as f:
        json.dump(run_results, f, indent=2)

    all_run_results.extend(run_results)

print(f"\nCompleted {len(all_run_results)} model-run entries.")



Starting Run 1/3 | Seed: 42

--- Training vgg16 ---


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG16_Weights.IMAGENET1K_V1`. You can also use `weights=VGG16_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


✅ Resumed from checkpoint

📊 VGG16 TEST RESULTS
Accuracy      : 0.9894
Top-5 Acc     : 0.9986
Macro F1      : 0.9893
Weighted F1   : 0.9893

--- Training vgg19 ---


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=VGG19_Weights.IMAGENET1K_V1`. You can also use `weights=VGG19_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


[vgg19] Epoch 1/25 | Train Loss: 5.4628 | Train Acc: 0.0171 | Val Loss: 4.6270 | Val Acc: 0.0868


[vgg19] Epoch 2/25 | Train Loss: 3.7233 | Train Acc: 0.2528 | Val Loss: 2.2884 | Val Acc: 0.6404


... (365 outputs left out)

[efficientnetb0] Epoch 22/25 | Train Loss: 0.9662 | Train Acc: 0.9994 | Val Loss: 1.0093 | Val Acc: 0.9909


[efficientnetb0] Epoch 23/25 | Train Loss: 0.9639 | Train Acc: 0.9997 | Val Loss: 1.0076 | Val Acc: 0.9909


[efficientnetb0] Epoch 24/25 | Train Loss: 0.9614 | Train Acc: 0.9993 | Val Loss: 1.0105 | Val Acc: 0.9915


[efficientnetb0] Epoch 25/25 | Train Loss: 0.9595 | Train Acc: 0.9991 | Val Loss: 1.0138 | Val Acc: 0.9921

📊 EFFICIENTNETB0 TEST RESULTS
Accuracy      : 0.9928
Top-5 Acc     : 0.9977
Macro F1      : 0.9928
Weighted F1   : 0.9928

Completed 15 model-run entries.


In [5]:
import pandas as pd
import numpy as np

metric_cols = [
    "Top-1 Accuracy",
    "Top-5 Accuracy",
    "Precision",
    "Recall",
    "Macro F1",
    "Weighted F1",
    "Training Time",
    "Best Epoch",
]

results_df = pd.DataFrame(all_run_results)
results_df = results_df[[
    "Run Number", "Random Seed", "Model",
    "Top-1 Accuracy", "Top-5 Accuracy",
    "Precision", "Recall", "Macro F1", "Weighted F1",
    "Training Time", "Best Epoch",
]]

summary_rows = []
for model_name in results_df["Model"].unique():
    model_df = results_df[results_df["Model"] == model_name]
    for col in metric_cols:
        mean_val = model_df[col].mean()
        std_val = model_df[col].std(ddof=1) if len(model_df) > 1 else 0.0
        summary_rows.append({
            "Model": model_name,
            "Metric": col,
            "Mean ± Standard Deviation": f"{mean_val:.6f} ± {std_val:.6f}",
            "Mean": mean_val,
            "Standard Deviation": std_val,
        })

summary_df = pd.DataFrame(summary_rows)

csv_path = os.path.join(OUTPUT_DIR, "summary_results.csv")
xlsx_path = os.path.join(OUTPUT_DIR, "summary_results.xlsx")

with open(csv_path, "w", encoding="utf-8") as f:
    f.write("Per-Run Results\n")
    results_df.to_csv(f, index=False)
    f.write("\nMean ± Standard Deviation (per model)\n")
    summary_df[["Model", "Metric", "Mean ± Standard Deviation"]].to_csv(f, index=False)

with pd.ExcelWriter(xlsx_path, engine="openpyxl") as writer:
    results_df.to_excel(writer, sheet_name="Per-Run Results", index=False)
    summary_df[["Model", "Metric", "Mean ± Standard Deviation"]].to_excel(
        writer, sheet_name="Mean ± Std", index=False
    )

print("Saved:", csv_path)
print("Saved:", xlsx_path)
print("\nPer-Run Results:")
print(results_df.to_string(index=False))
print("\nMean ± Standard Deviation:")
print(summary_df[["Model", "Metric", "Mean ± Standard Deviation"]].to_string(index=False))


Saved: multirun_baseline\summary_results.csv
Saved: multirun_baseline\summary_results.xlsx

Per-Run Results:
 Run Number  Random Seed          Model  Top-1 Accuracy  Top-5 Accuracy  Precision   Recall  Macro F1  Weighted F1  Training Time  Best Epoch
          1           42          vgg16        0.989386        0.998566   0.990531 0.989360  0.989320     0.989332       0.000000          23
          1           42          vgg19        0.993115        0.999139   0.993671 0.993117  0.993097     0.993095    7347.515344          24
          1           42    densenet121        0.994550        0.998566   0.995127 0.994551  0.994512     0.994510    3293.095158          25
          1           42    mobilenetv2        0.989673        0.998279   0.990835 0.989676  0.989567     0.989578    1611.300042          19
          1           42 efficientnetb0        0.990534        0.997705   0.991439 0.990536  0.990527     0.990524    2076.649615          19
          2          123          vgg16

In [6]:
import torch
import torch.nn as nn
from torchvision import models

def count_parameters(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = total - trainable
    return total, trainable, non_trainable

NUM_CLASSES_PREVIEW = 317  # same as training

models_dict = {
    "VGG16": models.vgg16(pretrained=False),
    "VGG19": models.vgg19(pretrained=False),
    "DenseNet121": models.densenet121(pretrained=False),
    "MobileNetV2": models.mobilenet_v2(pretrained=False),
    "EfficientNetB0": models.efficientnet_b0(pretrained=False),
}

models_dict["VGG16"].classifier[6] = nn.Linear(4096, NUM_CLASSES_PREVIEW)
models_dict["VGG19"].classifier[6] = nn.Linear(4096, NUM_CLASSES_PREVIEW)
models_dict["DenseNet121"].classifier = nn.Linear(1024, NUM_CLASSES_PREVIEW)
models_dict["MobileNetV2"].classifier[1] = nn.Linear(1280, NUM_CLASSES_PREVIEW)
models_dict["EfficientNetB0"].classifier[1] = nn.Linear(1280, NUM_CLASSES_PREVIEW)

print(f"{'Model':<18} {'Total Params':>15} {'Trainable':>15} {'Non-trainable':>15}")
print("-"*65)

for name, model in models_dict.items():
    total, trainable, non_trainable = count_parameters(model)
    print(f"{name:<18} {total:>15,} {trainable:>15,} {non_trainable:>15,}")


g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
g:\Ragib\ragib_torch_env\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=None`.
  warnings.warn(msg)


Model                 Total Params       Trainable   Non-trainable
-----------------------------------------------------------------
VGG16                  135,559,293     135,559,293               0
VGG19                  140,868,989     140,868,989               0
DenseNet121              7,278,781       7,278,781               0
MobileNetV2              2,629,949       2,629,949               0
EfficientNetB0           4,413,625       4,413,625               0
